# TrackLink USBL — Bearing Convention Investigation

Investigates whether the logged `target_bearing_angle` field is an
**absolute compass bearing** (degrees from North, clockwise) or a
**vessel-relative bearing** (degrees from the ship bow, clockwise).

Plots:
1. Ship track on an interactive map.
2. Time series of bearing, ship heading, bearing − heading, and slant range.
3. Simplified 2D target positions under both bearing interpretations overlaid
   on the same map.

The 2D estimate ignores target depth and uses `target_slant_range` directly
as horizontal range.  The two interpretations differ only in whether
`ship_heading` is added to the bearing before projecting.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pymap3d
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
DEPLOYMENT: str = "qdchdmy1_20130406_081713"

# Available raw parsed TrackLink deployments:
# qdch0ftq_20100428_020202
# qdch0ftq_20110415_020103
# qdch0ftq_20120430_002423
# qdch0ftq_20130406_023610
# qdchdmy1_20110416_005411
# qdchdmy1_20120501_071203
# qdchdmy1_20130406_081713

DATA_DIR: Path = Path("/data/exos_01/acfr_messages_v2_parsed")
USBL_FILE: Path = DATA_DIR / f"{DEPLOYMENT}_usbl_tracklink.csv"

## Load data

In [ ]:
REQUIRED_COLS: list[str] = [
    "timestamp",
    "ship_latitude",
    "ship_longitude",
    "ship_heading",
    "ship_roll",
    "ship_pitch",
    "target_bearing_angle",
    "target_slant_range",
]

usbl: pd.DataFrame = pd.read_csv(
    USBL_FILE, parse_dates=["timestamp"], date_format="ISO8601"
)

missing: list[str] = [col for col in REQUIRED_COLS if col not in usbl.columns]
if missing:
    raise ValueError(f"CSV is missing required columns: {missing}")

print(f"Rows: {len(usbl)}")
usbl.head(3)

## Ship track map

In [ ]:
t_s: np.ndarray = (usbl["timestamp"].astype(np.int64) / 1e9).to_numpy()
elapsed: np.ndarray = (t_s - t_s.min()) / 60.0

center_lat: float = float(usbl["ship_latitude"].mean())
center_lon: float = float(usbl["ship_longitude"].mean())
colorscale: str = "Viridis"

fig_map = go.Figure()

fig_map.add_trace(
    go.Scattermapbox(
        lat=usbl["ship_latitude"],
        lon=usbl["ship_longitude"],
        mode="lines+markers",
        marker=dict(
            size=5,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=True,
            colorbar=dict(title="Elapsed (min)", x=0.92),
        ),
        line=dict(width=1, color="royalblue"),
        name="Ship track",
        hovertemplate=(
            "<b>Ship</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "<extra></extra>"
        ),
    )
)

fig_map.update_layout(
    title=f"Ship track — {DEPLOYMENT}",
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=center_lat, lon=center_lon),
        zoom=13,
    ),
    legend=dict(x=0.01, y=0.99),
    height=600,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig_map.show()

## Time series: bearing, ship heading, and slant range

**Diagnostic:** for a roughly stationary target:
- If bearing is **absolute**, it stays ~constant while the ship turns — bearing and
  ship heading are *uncorrelated*.
- If bearing is **vessel-relative**, it shifts by the same amount as the ship
  turns — bearing and ship heading *track each other*.

The *bearing − ship heading* row makes this visible: it should be stable under
the relative interpretation and variable under the absolute interpretation.

In [ ]:
bearing_minus_heading: np.ndarray = (
    usbl["target_bearing_angle"].to_numpy()
    - usbl["ship_heading"].to_numpy()
    + 360.0
) % 360.0

fig_ts = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Target bearing angle and ship heading (°)",
        "Bearing − ship heading (°)",
        "Target slant range (m)",
    ),
    vertical_spacing=0.08,
)

fig_ts.add_trace(
    go.Scatter(
        x=usbl["timestamp"],
        y=usbl["target_bearing_angle"],
        mode="lines",
        name="Bearing",
        line=dict(color="darkorange"),
    ),
    row=1,
    col=1,
)
fig_ts.add_trace(
    go.Scatter(
        x=usbl["timestamp"],
        y=usbl["ship_heading"],
        mode="lines",
        name="Ship heading",
        line=dict(color="steelblue", dash="dash"),
    ),
    row=1,
    col=1,
)

fig_ts.add_trace(
    go.Scatter(
        x=usbl["timestamp"],
        y=bearing_minus_heading,
        mode="lines",
        name="Bearing − heading",
        line=dict(color="seagreen"),
    ),
    row=2,
    col=1,
)

fig_ts.add_trace(
    go.Scatter(
        x=usbl["timestamp"],
        y=usbl["target_slant_range"],
        mode="lines",
        name="Slant range",
        line=dict(color="crimson"),
    ),
    row=3,
    col=1,
)

fig_ts.update_layout(
    title=f"TrackLink USBL time series — {DEPLOYMENT}",
    height=750,
    legend=dict(x=0.01, y=0.99),
)

fig_ts.show()

## Simplified 2D target position estimates

Both estimates use `target_slant_range` as horizontal range (depth ignored)
and project from the ship GPS position via `pymap3d.ned2geodetic`.

| Interpretation | True bearing used |
|---|---|
| **Absolute** | `target_bearing_angle` directly |
| **Relative** | `(target_bearing_angle + ship_heading) % 360` |

In [ ]:
ship_lat: np.ndarray = usbl["ship_latitude"].to_numpy()
ship_lon: np.ndarray = usbl["ship_longitude"].to_numpy()
bearing: np.ndarray = usbl["target_bearing_angle"].to_numpy()
heading: np.ndarray = usbl["ship_heading"].to_numpy()
slant_range: np.ndarray = usbl["target_slant_range"].to_numpy()

zeros: np.ndarray = np.zeros_like(slant_range)


def project_target(
    true_bearing_deg: np.ndarray,
    horizontal_range_m: np.ndarray,
    ref_lat: np.ndarray,
    ref_lon: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Project horizontal range + true bearing to geodetic lat/lon."""
    bearing_rad: np.ndarray = np.radians(true_bearing_deg)
    north: np.ndarray = horizontal_range_m * np.cos(bearing_rad)
    east: np.ndarray = horizontal_range_m * np.sin(bearing_rad)
    target_lat: np.ndarray
    target_lon: np.ndarray
    target_lat, target_lon, _ = pymap3d.ned2geodetic(
        north, east, zeros, ref_lat, ref_lon, zeros
    )
    return target_lat, target_lon


target_lat_absolute: np.ndarray
target_lon_absolute: np.ndarray
target_lat_absolute, target_lon_absolute = project_target(
    bearing, slant_range, ship_lat, ship_lon
)

target_lat_relative: np.ndarray
target_lon_relative: np.ndarray
target_lat_relative, target_lon_relative = project_target(
    (bearing + heading) % 360.0, slant_range, ship_lat, ship_lon
)

usbl = usbl.assign(
    target_lat_absolute=target_lat_absolute,
    target_lon_absolute=target_lon_absolute,
    target_lat_relative=target_lat_relative,
    target_lon_relative=target_lon_relative,
)

print(
    "Absolute interpretation — target lat range:",
    f"{target_lat_absolute.min():.6f} to {target_lat_absolute.max():.6f}",
)
print(
    "Relative interpretation — target lat range:",
    f"{target_lat_relative.min():.6f} to {target_lat_relative.max():.6f}",
)

## Comparison map: absolute vs. relative bearing interpretation

In [ ]:
all_lats: np.ndarray = np.concatenate(
    [target_lat_absolute, target_lat_relative, ship_lat]
)
all_lons: np.ndarray = np.concatenate(
    [target_lon_absolute, target_lon_relative, ship_lon]
)
center_lat_cmp: float = float(all_lats.mean())
center_lon_cmp: float = float(all_lons.mean())

fig_cmp = go.Figure()

fig_cmp.add_trace(
    go.Scattermapbox(
        lat=usbl["ship_latitude"],
        lon=usbl["ship_longitude"],
        mode="lines",
        line=dict(width=1, color="royalblue"),
        name="Ship track",
        hovertemplate=(
            "<b>Ship</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "<extra></extra>"
        ),
    )
)

fig_cmp.add_trace(
    go.Scattermapbox(
        lat=usbl["target_lat_absolute"],
        lon=usbl["target_lon_absolute"],
        mode="markers",
        marker=dict(size=5, color="darkorange", opacity=0.7),
        name="Absolute bearing",
        customdata=np.stack(
            [usbl["target_bearing_angle"], usbl["target_slant_range"]], axis=1
        ),
        hovertemplate=(
            "<b>Absolute</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "Bearing: %{customdata[0]:.1f}°<br>"
            "Range: %{customdata[1]:.1f} m<br>"
            "<extra></extra>"
        ),
    )
)

fig_cmp.add_trace(
    go.Scattermapbox(
        lat=usbl["target_lat_relative"],
        lon=usbl["target_lon_relative"],
        mode="markers",
        marker=dict(size=5, color="seagreen", opacity=0.7),
        name="Relative bearing",
        customdata=np.stack(
            [
                usbl["target_bearing_angle"],
                usbl["ship_heading"],
                usbl["target_slant_range"],
            ],
            axis=1,
        ),
        hovertemplate=(
            "<b>Relative</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "Bearing: %{customdata[0]:.1f}°<br>"
            "Heading: %{customdata[1]:.1f}°<br>"
            "Range: %{customdata[2]:.1f} m<br>"
            "<extra></extra>"
        ),
    )
)

fig_cmp.update_layout(
    title=(
        f"2D target estimates: absolute vs. relative bearing — {DEPLOYMENT}"
    ),
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=center_lat_cmp, lon=center_lon_cmp),
        zoom=13,
    ),
    legend=dict(x=0.01, y=0.99),
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig_cmp.show()